# 15.1 — Prepare Gold Labels

## 0. Notebook Purpose

This notebook prepares the **real, manually curated gold-label source of truth** for the
*Signal Fusion with Meta-Learners* thesis pipeline.

### Label policy

| Label type | Status | Written to disk here? |
|---|---|---|
| Real manual labels (`manual_hardcoded`) | ✓ Included | ✓ Yes |
| Dummy / filler labels (`DUMMY_FILLER`) | ✗ Excluded | ✗ No |
| SMOTENC synthetic rows (`SYNTH_*`) | ✗ Excluded | ✗ No |

SMOTENC-based support augmentation is generated **in memory** inside Stage 2 episode
sampling (`sample_episode_with_synthetic_support`) and is never written to any CSV.

### Outputs

| File | Contents | Consumers |
|---|---|---|
| `gold_labels_clean.csv` | One row per manually labeled contract | Audit, Stage 2b eligibility |
| `contract_with_features_labeled_with_gold.csv` | Full feature matrix with `gold_y` joined in | Stage 1 (B/C/D), Stage 2 |

### How to add new labels

1. Append a new dict to `MANUAL_GOLD_LABELS` in **Section 2**.
2. Re-run the notebook end-to-end.
3. Both output CSVs are regenerated automatically.

## 1. Imports and Configuration

In [1]:
from pathlib import Path
from datetime import date

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

# ── Paths ──────────────────────────────────────────────────────────────────────
PROJECT_ROOT  = Path("..").resolve()
DATA_DIR      = PROJECT_ROOT / "Data"
PROCESSED_DIR = DATA_DIR / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_FILE            = PROCESSED_DIR / "contract_with_features_labeled.csv"
GOLD_LABELS_CLEAN_FILE  = PROCESSED_DIR / "gold_labels_clean.csv"
FEATURES_WITH_GOLD_FILE = PROCESSED_DIR / "contract_with_features_labeled_with_gold.csv"

# ── Column constants ───────────────────────────────────────────────────────────
CONTRACT_ID_COL  = "contract_id"
DEPARTMENT_COL   = "department"
GOLD_COL         = "gold_y"
LABEL_SOURCE_COL = "label_source"

print(f"Project root : {PROJECT_ROOT}")
print(f"Feature file : {FEATURE_FILE.name}  (exists: {FEATURE_FILE.exists()})")
print(f"Output 1     : {GOLD_LABELS_CLEAN_FILE.name}")
print(f"Output 2     : {FEATURES_WITH_GOLD_FILE.name}")

Project root : /Users/Thomas/Desktop/Master Thesis
Feature file : contract_with_features_labeled.csv  (exists: True)
Output 1     : gold_labels_clean.csv
Output 2     : contract_with_features_labeled_with_gold.csv


## 2. Manual Gold-Label Registry

This list is the **single authoritative place** where real, expert-reviewed labels are stored.
Each entry requires:

| Field | Type | Description |
|---|---|---|
| `contract_id` | `str` | Primary key from the contract database |
| `department` | `str` | Procurement department — must match the feature dataset exactly |
| `gold_y` | `int` | `1` = renegotiation recommended, `0` = not recommended |
| `label_source` | `str` | Always `"manual_hardcoded"` for real labels |
| `label_note` | `str` | Free-text provenance note (reviewer, rationale, date) |

> **Do not add** dummy, synthetic, or model-generated entries here.  
> Logistics labels are included — they are excluded from Stage 1 training
> downstream (by the Stage 1 leakage guard) but are the primary evaluation
> pool for Stage 2.

**Current inventory:** 33 labels across 5 departments  
(Logistics 13 · Packaging Material 8 · Global Strategic Outsourcing & Devices 5 ·
Bioprocessing and Excipients 4 · Aseptic Production 3)

In [2]:
# fmt: off
MANUAL_GOLD_LABELS = [
    # ── Logistics (meta-test target — Stage 2 only, excluded from Stage 1 training) ──
    {"contract_id": "192312", "department": "Logistics", "gold_y": 1, "label_source": "manual_hardcoded", "label_note": ""},
    {"contract_id": "541883", "department": "Logistics", "gold_y": 1, "label_source": "manual_hardcoded", "label_note": ""},
    {"contract_id": "582913", "department": "Logistics", "gold_y": 1, "label_source": "manual_hardcoded", "label_note": ""},
    {"contract_id": "569028", "department": "Logistics", "gold_y": 1, "label_source": "manual_hardcoded", "label_note": ""},
    {"contract_id": "470427", "department": "Logistics", "gold_y": 1, "label_source": "manual_hardcoded", "label_note": ""},
    {"contract_id": "530290", "department": "Logistics", "gold_y": 1, "label_source": "manual_hardcoded", "label_note": ""},
    {"contract_id": "533151", "department": "Logistics", "gold_y": 0, "label_source": "manual_hardcoded", "label_note": ""},
    {"contract_id": "550918", "department": "Logistics", "gold_y": 0, "label_source": "manual_hardcoded", "label_note": ""},
    {"contract_id": "525275", "department": "Logistics", "gold_y": 0, "label_source": "manual_hardcoded", "label_note": ""},
    {"contract_id": "335471", "department": "Logistics", "gold_y": 0, "label_source": "manual_hardcoded", "label_note": ""},
    {"contract_id": "607710", "department": "Logistics", "gold_y": 0, "label_source": "manual_hardcoded", "label_note": ""},
    {"contract_id": "536891", "department": "Logistics", "gold_y": 0, "label_source": "manual_hardcoded", "label_note": ""},
    {"contract_id": "599608", "department": "Logistics", "gold_y": 0, "label_source": "manual_hardcoded", "label_note": ""},
    # ── Packaging Material ──────────────────────────────────────────────────────
    {"contract_id": "193",  "department": "Packaging Material", "gold_y": 1, "label_source": "manual_hardcoded", "label_note": ""},
    {"contract_id": "213",  "department": "Packaging Material", "gold_y": 1, "label_source": "manual_hardcoded", "label_note": ""},
    {"contract_id": "190",  "department": "Packaging Material", "gold_y": 1, "label_source": "manual_hardcoded", "label_note": ""},
    {"contract_id": "1265", "department": "Packaging Material", "gold_y": 1, "label_source": "manual_hardcoded", "label_note": ""},
    {"contract_id": "1245", "department": "Packaging Material", "gold_y": 1, "label_source": "manual_hardcoded", "label_note": ""},
    {"contract_id": "1256", "department": "Packaging Material", "gold_y": 1, "label_source": "manual_hardcoded", "label_note": ""},
    {"contract_id": "248",  "department": "Packaging Material", "gold_y": 0, "label_source": "manual_hardcoded", "label_note": ""},
    {"contract_id": "250",  "department": "Packaging Material", "gold_y": 0, "label_source": "manual_hardcoded", "label_note": ""},
    # ── Global Strategic Outsourcing & Devices ──────────────────────────────────
    {"contract_id": "4821",   "department": "Global Strategic Outsourcing & Devices", "gold_y": 1, "label_source": "manual_hardcoded", "label_note": ""},
    {"contract_id": "557",    "department": "Global Strategic Outsourcing & Devices", "gold_y": 1, "label_source": "manual_hardcoded", "label_note": ""},
    {"contract_id": "565",    "department": "Global Strategic Outsourcing & Devices", "gold_y": 1, "label_source": "manual_hardcoded", "label_note": ""},
    {"contract_id": "483149", "department": "Global Strategic Outsourcing & Devices", "gold_y": 0, "label_source": "manual_hardcoded", "label_note": ""},
    {"contract_id": "25777",  "department": "Global Strategic Outsourcing & Devices", "gold_y": 0, "label_source": "manual_hardcoded", "label_note": ""},
    # ── Bioprocessing and Excipients ────────────────────────────────────────────
    {"contract_id": "504361", "department": "Bioprocessing and Excipients", "gold_y": 1, "label_source": "manual_hardcoded", "label_note": ""},
    {"contract_id": "613927", "department": "Bioprocessing and Excipients", "gold_y": 1, "label_source": "manual_hardcoded", "label_note": ""},
    {"contract_id": "345318", "department": "Bioprocessing and Excipients", "gold_y": 0, "label_source": "manual_hardcoded", "label_note": ""},
    {"contract_id": "1877",   "department": "Bioprocessing and Excipients", "gold_y": 0, "label_source": "manual_hardcoded", "label_note": ""},
    # ── Aseptic Production ──────────────────────────────────────────────────────
    {"contract_id": "577893", "department": "Aseptic Production", "gold_y": 0, "label_source": "manual_hardcoded", "label_note": ""},
    {"contract_id": "370849", "department": "Aseptic Production", "gold_y": 0, "label_source": "manual_hardcoded", "label_note": ""},
    {"contract_id": "595207", "department": "Aseptic Production", "gold_y": 0, "label_source": "manual_hardcoded", "label_note": ""},
]
# fmt: on

print(f"Total entries in MANUAL_GOLD_LABELS: {len(MANUAL_GOLD_LABELS)}")

Total entries in MANUAL_GOLD_LABELS: 33


## 3. Normalize and Validate the Registry

Before the registry is used downstream, three normalization and four validation
checks are applied:

**Normalization**
- `contract_id` and `department` are cast to string and stripped (removes trailing
  `.0` introduced by float conversion when reading from Excel or CSV).
- `gold_y` is coerced to nullable integer (`Int64`).

**Validation**
1. `gold_y` must be exactly `0` or `1` — no `NaN`, no floats.
2. No duplicate `(contract_id, department)` pairs.
3. No single `contract_id` assigned to more than one department.
4. `label_source` must be `"manual_hardcoded"` for every row.

In [3]:
df_gold = pd.DataFrame(MANUAL_GOLD_LABELS)

# ── Normalize string keys ──────────────────────────────────────────────────────
for col in [CONTRACT_ID_COL, DEPARTMENT_COL]:
    df_gold[col] = (
        df_gold[col]
        .astype("string")
        .str.replace(r"\.0$", "", regex=True)
        .str.strip()
    )

# ── Normalize gold_y ───────────────────────────────────────────────────────────
df_gold[GOLD_COL] = pd.to_numeric(df_gold[GOLD_COL], errors="coerce")

invalid_labels = df_gold[
    df_gold[GOLD_COL].isna() | ~df_gold[GOLD_COL].isin([0, 1])
]
if not invalid_labels.empty:
    raise ValueError(
        f"Invalid gold_y values found (must be 0 or 1):\n{invalid_labels}"
    )

df_gold[GOLD_COL] = df_gold[GOLD_COL].astype("Int64")

# ── Validation 1: no duplicate (contract_id, department) pairs ─────────────────
dup_pairs = df_gold[
    df_gold.duplicated(subset=[CONTRACT_ID_COL, DEPARTMENT_COL], keep=False)
]
if not dup_pairs.empty:
    raise ValueError(
        f"Duplicate (contract_id, department) pairs in MANUAL_GOLD_LABELS:\n{dup_pairs}"
    )

# ── Validation 2: no contract_id spanning multiple departments ─────────────────
multi_dept = (
    df_gold.groupby(CONTRACT_ID_COL)[DEPARTMENT_COL]
    .nunique()
    .pipe(lambda s: s[s > 1])
)
if not multi_dept.empty:
    raise ValueError(
        f"These contract_ids appear in more than one department:\n{multi_dept}"
    )

# ── Validation 3: only manual_hardcoded label sources ─────────────────────────
if LABEL_SOURCE_COL in df_gold.columns:
    bad_sources = df_gold[df_gold[LABEL_SOURCE_COL] != "manual_hardcoded"]
    if not bad_sources.empty:
        raise ValueError(
            f"Non-manual label sources found — only 'manual_hardcoded' is "
            f"permitted in MANUAL_GOLD_LABELS:\n"
            f"{bad_sources[[CONTRACT_ID_COL, DEPARTMENT_COL, LABEL_SOURCE_COL]]}"
        )

print(f"\u2713 Validation passed")
print(f"  Total labels    : {len(df_gold)}")
print(f"  Departments     : {sorted(df_gold[DEPARTMENT_COL].unique().tolist())}")
display(df_gold)

✓ Validation passed
  Total labels    : 33
  Departments     : ['Aseptic Production', 'Bioprocessing and Excipients', 'Global Strategic Outsourcing & Devices', 'Logistics', 'Packaging Material']


,contract_id,department,gold_y,label_source,label_note
0,192312,Logistics,1,manual_hardcoded,
1,541883,Logistics,1,manual_hardcoded,
2,582913,Logistics,1,manual_hardcoded,
3,569028,Logistics,1,manual_hardcoded,
4,470427,Logistics,1,manual_hardcoded,
5,530290,Logistics,1,manual_hardcoded,
6,533151,Logistics,0,manual_hardcoded,
7,550918,Logistics,0,manual_hardcoded,
8,525275,Logistics,0,manual_hardcoded,
9,335471,Logistics,0,manual_hardcoded,


## 4. Coverage Report

The table below shows the **real label count per department and class**.
Not every department has both positive and negative examples.

Downstream, `filter_valid_departments()` (in `episode_sampler.py`) rejects any
department that cannot form a support set of at least `k_pos` positive and
`k_neg` negative labeled contracts. The real-label pool therefore sets the
upper bound on which departments can participate in meta-training and which
values of `k` are feasible.

> **Logistics** is the held-out meta-test target. Its labels are excluded from
> Stage 1 training by the leakage guard in `fit_stage1_conditions()` and from
> meta-training by the `target_department != dept` filter in Stage 2. They are
> used exclusively for Stage 2 evaluation episodes.

In [4]:
coverage = (
    df_gold.groupby(DEPARTMENT_COL)[GOLD_COL]
    .agg(
        real_yes=lambda s: int((s == 1).sum()),
        real_no=lambda s:  int((s == 0).sum()),
        real_total="count",
    )
    .reset_index()
    .sort_values("real_total", ascending=False)
    .reset_index(drop=True)
)

print("Real gold-label coverage per department (thesis mode \u2014 no dummy fill):")
display(coverage)
print(f"\nTotal: {coverage['real_total'].sum()} real labels across {len(coverage)} departments")

Real gold-label coverage per department (thesis mode — no dummy fill):


,department,real_yes,real_no,real_total
0,Logistics,6,7,13
1,Packaging Material,6,2,8
2,Global Strategic Outsourcing & Devices,3,2,5
3,Bioprocessing and Excipients,2,2,4
4,Aseptic Production,0,3,3



Total: 33 real labels across 5 departments


## 5. Load Feature Matrix

The base feature dataset is the output of the Snorkel notebook
(`contract_with_features_labeled.csv`). It contains all contracts with
their weak supervision probability scores (`renegotiation_prob`). Gold labels
are joined onto this file to produce the final modeling input.

Both join-key columns (`contract_id`, `department`) are normalized here to the
same string format used in the gold-label registry.

In [5]:
if not FEATURE_FILE.exists():
    raise FileNotFoundError(
        f"Feature file not found: {FEATURE_FILE}\n"
        "Run the Snorkel notebook first to generate this file."
    )

df_features = pd.read_csv(FEATURE_FILE, low_memory=False)

# Normalize join keys to match gold-label representation
for col in [CONTRACT_ID_COL, DEPARTMENT_COL]:
    if col in df_features.columns:
        df_features[col] = (
            df_features[col]
            .astype("string")
            .str.replace(r"\.0$", "", regex=True)
            .str.strip()
        )

# Drop rows with missing contract_id (cannot be labeled reliably)
df_features = df_features[df_features[CONTRACT_ID_COL].notna()].copy()

print(f"Feature matrix loaded : {df_features.shape}")
print(f"Unique contract_ids   : {df_features[CONTRACT_ID_COL].nunique()}")
print(f"Unique departments    : {df_features[DEPARTMENT_COL].nunique()}")
print(f"Has renegotiation_prob: {'renegotiation_prob' in df_features.columns}")
display(df_features.head(3))

Feature matrix loaded : (9201, 165)
Unique contract_ids   : 2209
Unique departments    : 14
Has renegotiation_prob: True


,contract_id,contract_name,contract_status,contract_owner_cost_centre,terminated,term_type,start_date,expiration_date,supplier_id,supplier_number,supplier_display_name,payment_terms,incoterms,contract_currency_code,contract_value,contract_value_dkk,contract_type,nn_contract_type,contract_commodity,team,unit,company_country,start_year,expiry_year,open_ended_contract,end_year,start_year_capped,observation_year,contract_age_years,expiry_pressure_bucket,department_from_cost_center,department,moodys_bvd_id,fin_moodys_company_name,fin_closing_date,fin_created_at_utc,fin_Status,fin_implied_rating,fin_risk_level,fin_financial_level,fin_output_text,fin_Implied_rating - original,fin_Number_of_months,fin_Total_shareholders'_funds_and_liabilitiesth_DKK,fin_profit_margin,fin_EBITDA_margin,fin_ebit_margin,fin_current_ratio,fin_gearing,fin_Cash_ratio,fin_long_term_gearing,fin_Total_liabilities_Equity_ratio,fin_short_term_liabilities_equity_ratio,fin_interest_coverage_ratio,fin_solvency_ratio_asset_based,fin_debt_asset_ratio,fin_roe_net_income,fin_roa_net_income,fin_Net_assets_turnover,fin_Number_of_employees,fin_financial_risk_score,esg_esg_overall,esg_esg_industry_adjusted,esg_env_score,esg_env_weight,esg_social_score,esg_social_weight,esg_gov_score,esg_gov_weight,esg_industry_min,esg_industry_max,news_article_count,news_negative_count,news_positive_count,news_neutral_count,news_avg_sentiment_score,news_avg_relevance_score,news_max_relevance_score,news_negative_ratio,news_has_high_relevance_negative_news,Company name Latin alphabet,avg_vol,vol_stability_score,vol_shock_ratio,vol_trend_slope,Risk level,market_cap_volatility,supplier_sector,moodys_risk_rating,Price_trends_52 weeks_%,Earnings_per_share_DKK,Shares outstanding,Current_market_capitalisation_DKK,avg_closing_price,price_volatility_score,price_trend_slope,Risk level_stock_closing_price,company_country_clean,Country,Code,LPI_Score,Customs,Infrastructure,International_Shipments,Tracking_Tracing,Timeliness,PPI_Value,gold_y,gold_department,fin_implied_rating_missing_flag,fin_financial_level_missing_flag,fin_current_ratio_missing_flag,fin_interest_coverage_ratio_missing_flag,fin_ebit_margin_missing_flag,fin_implied_rating_ordinal,fin_flag_moderate_or_worse_rating,fin_flag_risk_take_caution_or_worse,fin_flag_risk_do_not_source,fin_flag_financial_risk_score_high,fin_flag_financial_risk_score_very_high,fin_flag_liquidity_stress,fin_flag_severe_liquidity_stress,fin_flag_strong_liquidity,fin_flag_gearing_high,fin_flag_long_term_gearing_high,fin_flag_short_term_gearing_high,fin_flag_debt_asset_high,fin_flag_long_term_liab_equity_high,fin_flag_short_term_liab_equity_high,fin_flag_interest_coverage_stress,fin_flag_interest_coverage_weak,fin_flag_low_solvency,fin_flag_very_low_solvency,fin_flag_negative_ebit_margin,fin_flag_negative_roe,fin_flag_negative_roa,fin_total_stress_flags,fin_flag_multiple_financial_stress_signals,fin_flag_severe_financial_stress,esg_below_industry_min,supplier_is_publicly_listed,market_flag_high_volume_shock,market_flag_high_market_cap_volatility,market_flag_negative_volume_trend,market_flag_negative_price_trend,market_flag_negative_52w_price_trend,market_flag_high_beta,market_flag_negative_eps,market_flag_stock_price_take_caution_or_worse,market_log_vol_shock_ratio,lpi_relative_risk,is_old_and_near_expiry,esg_row_missing_pct,news_row_missing_pct,lpi_below_supplier_median,years_to_expiry_capped,contracts_per_supplier,financial_data_available,esg_data_available,news_data_available,market_data_available,has_environmental_appendix,contract_name_lower,renegotiation_prob,target_renegotiate
0,9675,Bioreliance_Master_2018_MSA,published,7756.0,0,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,38367,BIORELIANCE LIMITED INVITROGEN BIOSERVICES,F030 Invoice Date + 30 days,DDP Delivered duty paid,GBP,0.0,0.0,Production,Master Service Agreement,Outsourced GMP Laboratory Analysis,"Quality, Production Services & Supplies",SSIMS,DENMARK,2018,2027,0,2025,2018,2018,0,5y_pl

## 6. Join Gold Labels onto Feature Matrix

Gold labels are joined using the **composite key `(contract_id, department)`**.
This is stricter than joining on `contract_id` alone: it prevents a label from
propagating to a row whose department does not match the intended labeled
department, which could occur if the same physical contract appears under two
department names in the feature table.

The join is a left join — all feature rows are retained. Rows without a matching
gold label receive `NaN` for `gold_y` and `label_source`. Stage 1 and Stage 2
filter on `gold_y.notna()` to select labeled rows.

Any stale gold columns from a previous run of this notebook are dropped before
the join to prevent column conflicts.

In [6]:
# Drop stale gold columns from any previous run
stale_cols = [GOLD_COL, LABEL_SOURCE_COL, "label_date", "label_note", "notes", "gold_department"]
df_features = df_features.drop(
    columns=[c for c in stale_cols if c in df_features.columns]
)

# Columns to carry over from the gold registry
gold_join_cols = [CONTRACT_ID_COL, DEPARTMENT_COL, GOLD_COL, LABEL_SOURCE_COL]
if "label_note" in df_gold.columns:
    gold_join_cols.append("label_note")

df_with_gold = df_features.merge(
    df_gold[gold_join_cols],
    on=[CONTRACT_ID_COL, DEPARTMENT_COL],
    how="left",
)

n_labeled = int(df_with_gold[GOLD_COL].notna().sum())
n_total   = len(df_with_gold)

print(f"Rows in joined file       : {n_total}")
print(f"Rows with gold_y          : {n_labeled}")
print(f"Rows without gold_y (NaN) : {n_total - n_labeled}")
print()
print("Class distribution among labeled rows:")
print(df_with_gold[GOLD_COL].value_counts(dropna=True).to_string())
display(df_with_gold[df_with_gold[GOLD_COL].notna()].head(5))

Rows in joined file       : 9201
Rows with gold_y          : 99
Rows without gold_y (NaN) : 9102

Class distribution among labeled rows:
gold_y
1    68
0    31


,contract_id,contract_name,contract_status,contract_owner_cost_centre,terminated,term_type,start_date,expiration_date,supplier_id,supplier_number,supplier_display_name,payment_terms,incoterms,contract_currency_code,contract_value,contract_value_dkk,contract_type,nn_contract_type,contract_commodity,team,unit,company_country,start_year,expiry_year,open_ended_contract,end_year,start_year_capped,observation_year,contract_age_years,expiry_pressure_bucket,department_from_cost_center,department,moodys_bvd_id,fin_moodys_company_name,fin_closing_date,fin_created_at_utc,fin_Status,fin_implied_rating,fin_risk_level,fin_financial_level,fin_output_text,fin_Implied_rating - original,fin_Number_of_months,fin_Total_shareholders'_funds_and_liabilitiesth_DKK,fin_profit_margin,fin_EBITDA_margin,fin_ebit_margin,fin_current_ratio,fin_gearing,fin_Cash_ratio,fin_long_term_gearing,fin_Total_liabilities_Equity_ratio,fin_short_term_liabilities_equity_ratio,fin_interest_coverage_ratio,fin_solvency_ratio_asset_based,fin_debt_asset_ratio,fin_roe_net_income,fin_roa_net_income,fin_Net_assets_turnover,fin_Number_of_employees,fin_financial_risk_score,esg_esg_overall,esg_esg_industry_adjusted,esg_env_score,esg_env_weight,esg_social_score,esg_social_weight,esg_gov_score,esg_gov_weight,esg_industry_min,esg_industry_max,news_article_count,news_negative_count,news_positive_count,news_neutral_count,news_avg_sentiment_score,news_avg_relevance_score,news_max_relevance_score,news_negative_ratio,news_has_high_relevance_negative_news,Company name Latin alphabet,avg_vol,vol_stability_score,vol_shock_ratio,vol_trend_slope,Risk level,market_cap_volatility,supplier_sector,moodys_risk_rating,Price_trends_52 weeks_%,Earnings_per_share_DKK,Shares outstanding,Current_market_capitalisation_DKK,avg_closing_price,price_volatility_score,price_trend_slope,Risk level_stock_closing_price,company_country_clean,Country,Code,LPI_Score,Customs,Infrastructure,International_Shipments,Tracking_Tracing,Timeliness,PPI_Value,fin_implied_rating_missing_flag,fin_financial_level_missing_flag,fin_current_ratio_missing_flag,fin_interest_coverage_ratio_missing_flag,fin_ebit_margin_missing_flag,fin_implied_rating_ordinal,fin_flag_moderate_or_worse_rating,fin_flag_risk_take_caution_or_worse,fin_flag_risk_do_not_source,fin_flag_financial_risk_score_high,fin_flag_financial_risk_score_very_high,fin_flag_liquidity_stress,fin_flag_severe_liquidity_stress,fin_flag_strong_liquidity,fin_flag_gearing_high,fin_flag_long_term_gearing_high,fin_flag_short_term_gearing_high,fin_flag_debt_asset_high,fin_flag_long_term_liab_equity_high,fin_flag_short_term_liab_equity_high,fin_flag_interest_coverage_stress,fin_flag_interest_coverage_weak,fin_flag_low_solvency,fin_flag_very_low_solvency,fin_flag_negative_ebit_margin,fin_flag_negative_roe,fin_flag_negative_roa,fin_total_stress_flags,fin_flag_multiple_financial_stress_signals,fin_flag_severe_financial_stress,esg_below_industry_min,supplier_is_publicly_listed,market_flag_high_volume_shock,market_flag_high_market_cap_volatility,market_flag_negative_volume_trend,market_flag_negative_price_trend,market_flag_negative_52w_price_trend,market_flag_high_beta,market_flag_negative_eps,market_flag_stock_price_take_caution_or_worse,market_log_vol_shock_ratio,lpi_relative_risk,is_old_and_near_expiry,esg_row_missing_pct,news_row_missing_pct,lpi_below_supplier_median,years_to_expiry_capped,contracts_per_supplier,financial_data_available,esg_data_available,news_data_available,market_data_available,has_environmental_appendix,contract_name_lower,renegotiation_prob,target_renegotiate,gold_y,label_source,label_note
194,541883,KUEHNE + NAGEL A/S_MASTER_2017_NNAS_Primary_LS...,published,9589.0,0,fixed,2022-01-01 00:00:00+00:00,2026-12-31 00:00:00+00:00,441297,162506,KUEHNE + NAGEL S.À R.L,F045 Invoice Date + 45 days,NaN,EUR,0.0,0.0,"Finance, Legal & Procurement",Logistics Services Provider Agreement,4PL,NaN,SSIMS,DENMARK,2022,2026,0,2025,2022,2022,0,3_to_5y,NaN,Logistics,NaN,NaN,NaN,NaN,NaN,NaN

## 7. Invariant Checks

Four assertions must pass before the outputs are written:

1. **No SYNTH_ IDs** — SMOTENC contract IDs must never appear in any gold-label column.
2. **No DUMMY_FILLER sources** — only `manual_hardcoded` is permitted.
3. **No illegal `gold_y` values** — only `0`, `1`, and `NaN` are valid.
4. **Coverage check** — every gold label in the registry was matched to at least
   one row in the feature matrix. Unmatched labels are reported as warnings
   (they appear in `gold_labels_clean.csv` but not in the joined file).

If any assertion fails, the notebook halts before writing to disk. Fix the
offending entry in `MANUAL_GOLD_LABELS` and re-run.

In [7]:
labeled_rows = df_with_gold[df_with_gold[GOLD_COL].notna()].copy()

# ── 1. No SYNTH_ IDs ───────────────────────────────────────────────────────────
synth_in_gold = labeled_rows[
    labeled_rows[CONTRACT_ID_COL].str.startswith("SYNTH_", na=False)
]
assert synth_in_gold.empty, (
    "SMOTENC SYNTH_ IDs found in the gold column — "
    "synthetic rows must never be written to this file."
)

# ── 2. No DUMMY_FILLER label sources ──────────────────────────────────────────
if LABEL_SOURCE_COL in labeled_rows.columns:
    bad_sources = labeled_rows[labeled_rows[LABEL_SOURCE_COL] != "manual_hardcoded"]
    assert bad_sources.empty, (
        "Non-manual label_source found — DUMMY_FILLER rows must not appear "
        "in the thesis-mode gold file:\n"
        + str(bad_sources[[CONTRACT_ID_COL, DEPARTMENT_COL, LABEL_SOURCE_COL]])
    )

# ── 3. No illegal gold_y values ────────────────────────────────────────────────
illegal_vals = df_with_gold[
    df_with_gold[GOLD_COL].notna() & ~df_with_gold[GOLD_COL].isin([0, 1])
]
assert illegal_vals.empty, (
    f"Illegal gold_y values (must be 0 or 1):\n{illegal_vals}"
)

# ── 4. Coverage: all registry IDs present in the feature matrix ────────────────
feature_ids  = set(df_with_gold[CONTRACT_ID_COL].dropna().astype(str))
registry_ids = set(df_gold[CONTRACT_ID_COL].dropna().astype(str))
unmatched    = registry_ids - feature_ids

if unmatched:
    unmatched_df = df_gold[df_gold[CONTRACT_ID_COL].isin(unmatched)]
    print(
        f"WARNING: {len(unmatched)} gold contract_id(s) not found in the "
        f"feature matrix. They appear in gold_labels_clean.csv but NOT "
        f"in the joined file (Stage 1 and Stage 2 will not see them):"
    )
    display(unmatched_df[[CONTRACT_ID_COL, DEPARTMENT_COL, GOLD_COL]])
else:
    print(f"\u2713 All {len(registry_ids)} gold contract_ids matched in the feature matrix.")

print()
print("\u2713 Invariants 1\u20133 passed.")
print(f"  Labeled rows in joined file : {n_labeled}")
print(f"  label_source values         : manual_hardcoded only")

,contract_id,department,gold_y
5,530290,Logistics,1
7,550918,Logistics,0
10,607710,Logistics,0
11,536891,Logistics,0
12,599608,Logistics,0
25,25777,Global Strategic Outsourcing & Devices,0
27,613927,Bioprocessing and Excipients,1
28,345318,Bioprocessing and Excipients,0
29,1877,Bioprocessing and Excipients,0
30,577893,Aseptic Production,0



✓ Invariants 1–3 passed.
  Labeled rows in joined file : 99
  label_source values         : manual_hardcoded only


## 8. Save Outputs

Two files are written:

| File | Contents |
|---|---|
| `gold_labels_clean.csv` | Compact registry — one row per manually labeled contract. Used for audit and Stage 2b k-shot eligibility analysis. |
| `contract_with_features_labeled_with_gold.csv` | Full feature matrix with `gold_y` joined in. Primary input for Stage 1 (conditions B/C/D) and Stage 2 episode sampling. |

Both files contain **only real labels**. No dummy or synthetic rows.

In [8]:
df_gold.to_csv(GOLD_LABELS_CLEAN_FILE, index=False)
df_with_gold.to_csv(FEATURES_WITH_GOLD_FILE, index=False)

print(f"Saved: {GOLD_LABELS_CLEAN_FILE.name}")
print(f"       {len(df_gold)} real gold labels")
print()
print(f"Saved: {FEATURES_WITH_GOLD_FILE.name}")
print(f"       {len(df_with_gold)} total rows, {n_labeled} with gold_y")

Saved: gold_labels_clean.csv
       33 real gold labels

Saved: contract_with_features_labeled_with_gold.csv
       9201 total rows, 99 with gold_y


## 9. Summary

In [9]:
summary = (
    df_gold.groupby(DEPARTMENT_COL)[GOLD_COL]
    .agg(
        real_yes=lambda s: int((s == 1).sum()),
        real_no=lambda s:  int((s == 0).sum()),
        real_total="count",
    )
    .reset_index()
    .sort_values("real_total", ascending=False)
    .reset_index(drop=True)
)
summary["stage1_eligible"] = summary[DEPARTMENT_COL] != "Logistics"
summary["note"] = summary[DEPARTMENT_COL].apply(
    lambda d: "meta-test target (Stage 2 only)" if d == "Logistics" else ""
)

print("=" * 65)
print("GOLD LABEL SUMMARY \u2014 THESIS MODE (real labels only)")
print("=" * 65)
display(summary)
print()
print(f"  Total real labels : {summary['real_total'].sum()}")
print(f"  Departments       : {len(summary)}")
print()
print("  Downstream usage:")
print(f"  \u251c\u2500 {GOLD_LABELS_CLEAN_FILE.name}")
print("  \u2502    \u2192 Stage 2b k-shot eligibility and coverage analysis")
print(f"  \u2514\u2500 {FEATURES_WITH_GOLD_FILE.name}")
print("       \u2192 Stage 1 conditions B_gold_only, C_hybrid, D_hybrid_unweighted")
print("       \u2192 Stage 2 support/query episode sampling")
print()
print("  Note: Logistics labels are filtered OUT of Stage 1 training")
print("        by the leakage guard in fit_stage1_conditions().")
print("        They are used ONLY in Stage 2 meta-test episodes.")

GOLD LABEL SUMMARY — THESIS MODE (real labels only)


,department,real_yes,real_no,real_total,stage1_eligible,note
0,Logistics,6,7,13,False,meta-test target (Stage 2 only)
1,Packaging Material,6,2,8,True,
2,Global Strategic Outsourcing & Devices,3,2,5,True,
3,Bioprocessing and Excipients,2,2,4,True,
4,Aseptic Production,0,3,3,True,



  Total real labels : 33
  Departments       : 5

  Downstream usage:
  ├─ gold_labels_clean.csv
  │    → Stage 2b k-shot eligibility and coverage analysis
  └─ contract_with_features_labeled_with_gold.csv
       → Stage 1 conditions B_gold_only, C_hybrid, D_hybrid_unweighted
       → Stage 2 support/query episode sampling

  Note: Logistics labels are filtered OUT of Stage 1 training
        by the leakage guard in fit_stage1_conditions().
        They are used ONLY in Stage 2 meta-test episodes.
